## 1.Problem Statement

Financial markets are dynamic and unpredictable. Traders often rely on intuition or basic indicators, which may fail to capture hidden patterns in historical price data.

This project aims to build a machine learning system that predicts whether a stock’s closing price will go UP or DOWN the next day using historical market data and technical indicators.

## 2.Objective

- Build a binary classification model to predict next-day stock direction (UP/DOWN)
- Use technical indicators: Moving Averages, RSI, MACD, Lag features
- Train an XGBoost model for tabular data
- Evaluate using Accuracy, Precision, Recall, F1 Score
- Save the trained model for backend deployment

## 3.Import require libraries

In [1]:
import pandas as pd
import numpy as np
import yfinance as yf

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

import pickle

## 4.Load data

In [9]:
def load_data(ticker, start="2020-01-01", end="2024-01-01"):
    df = yf.download(ticker, start=start, end=end)
    df.dropna(inplace=True)
    return df

df = load_data("RELIANCE.NS")
df.head(10)

[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume
Ticker,RELIANCE.NS,RELIANCE.NS,RELIANCE.NS,RELIANCE.NS,RELIANCE.NS
Date,,,,,
2020-01-01,675.324158,683.152852,673.490062,679.081936,14004468
2020-01-02,686.821228,689.348791,676.397899,676.397899,17710316
2020-01-03,687.648865,689.661956,681.318790,685.792313,20984698
2020-01-06,671.700684,683.510767,670.134933,679.976719,24519177
2020-01-07,682.034546,686.463335,677.068889,679.529321,16683622
2020-01-08,676.912415,686.441004,675.503262,677.740024,16047902
2020-01-09,692.502625,693.397305,685.009465,688.297506,14982742
2020-01-10,692.346008,696.953733,688.767166,694.247235,12478359


In [10]:
df.columns = df.columns.droplevel(1)
df.head()

Price,Close,High,Low,Open,Volume
Date,,,,,
2020-01-01,675.324158,683.152852,673.490062,679.081936,14004468
2020-01-02,686.821228,689.348791,676.397899,676.397899,17710316
2020-01-03,687.648865,689.661956,681.318790,685.792313,20984698
2020-01-06,671.700684,683.510767,670.134933,679.976719,24519177
2020-01-07,682.034546,686.463335,677.068889,679.529321,16683622


In [11]:
# Flatten columns completely
df.columns = df.columns.get_level_values(-1)

# Reset index so Date becomes a column
df = df.reset_index()

df.head()

Price,Date,Close,High,Low,Open,Volume
0,2020-01-01,675.324158,683.152852,673.490062,679.081936,14004468
1,2020-01-02,686.821228,689.348791,676.397899,676.397899,17710316
2,2020-01-03,687.648865,689.661956,681.318790,685.792313,20984698
3,2020-01-06,671.700684,683.510767,670.134933,679.976719,24519177
4,2020-01-07,682.034546,686.463335,677.068889,679.529321,16683622


## Target Creation

In [12]:
# Create target: 1 = UP, 0 = DOWN
df['Target'] = (df['Close'].shift(-1) > df['Close']).astype(int)

# Drop last row (no "tomorrow" to compare)
df = df[:-1]

df.head()

Price,Date,Close,High,Low,Open,Volume,Target
0,2020-01-01,675.324158,683.152852,673.490062,679.081936,14004468,1
1,2020-01-02,686.821228,689.348791,676.397899,676.397899,17710316,1
2,2020-01-03,687.648865,689.661956,681.318790,685.792313,20984698,0
3,2020-01-06,671.700684,683.510767,670.134933,679.976719,24519177,1
4,2020-01-07,682.034546,686.463335,677.068889,679.529321,16683622,0


## Feature Engineering

In [22]:
def add_features(df):
    # Moving Averages
    df['MA7'] = df['Close'].rolling(7).mean()
    df['MA21'] = df['Close'].rolling(21).mean()
    df['Volatility'] = df['Close'].rolling(7).std()

    # Daily Return
    df['Return'] = df['Close'].pct_change()

    # RSI
    delta = df['Close'].diff()
    gain = (delta.where(delta > 0, 0)).rolling(14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(14).mean()
    rs = gain / loss
    df['RSI'] = 100 - (100 / (1 + rs))

    # MACD
    ema12 = df['Close'].ewm(span=12).mean()
    ema26 = df['Close'].ewm(span=26).mean()
    df['MACD'] = ema12 - ema26

    # Lag Features
    df['Lag1'] = df['Close'].shift(1)
    df['Lag2'] = df['Close'].shift(2)

    df.dropna(inplace=True)
    return df


df = add_features(df)
df.head()

Price,Date,Close,High,Low,Open,Volume,Target,MA7,MA21,Return,RSI,MACD,Lag1,Lag2,Volatility
40,2020-02-27,620.143250,623.520759,612.426360,620.926065,25054993,0,645.258937,647.224098,-0.004131,35.817590,-2.904996,622.715515,633.631042,20.554754
41,2020-02-28,594.375671,607.058158,592.742849,606.118684,39315769,0,636.391750,644.003153,-0.041551,31.623986,-5.934354,620.143250,622.715515,27.229136
42,2020-03-02,588.783752,612.426315,582.051072,606.208105,29500495,1,624.399493,641.284950,-0.009408,29.839611,-8.593316,594.375671,620.143250,27.042993
43,2020-03-03,600.728149,605.671410,591.400834,594.845474,27398798,0,615.254316,639.819336,0.020287,32.030719,-9.735903,588.783752,594.375671,21.350735
44,2020-03-04,599.318909,605.626586,585.093068,604.821374,22105261,0,608.528041,638.843680,-0.002346,27.590918,-10.607893,600.728149,588.783752,16.843055


In [23]:
df[['Close', 'Target']].head(10)

Price,Close,Target
40,620.143250,0
41,594.375671,0
42,588.783752,1
43,600.728149,0
44,599.318909,0
45,586.547058,0
46,568.585754,0
47,498.418457,1
48,516.044189,0
49,475.536316,1


In [24]:
df['Target'].value_counts()

Target
1    501
0    450
Name: count, dtype: int64

## Train-Test Split

In [28]:
# Feature columns
features = ['MA7', 'MA21', 'RSI', 'MACD', 'Lag1', 'Lag2', 'Volume']

X = df[features]
y = df['Target']

# Time-aware split (NO shuffle)
split_index = int(len(df) * 0.8)

X_train, X_test = X[:split_index], X[split_index:]
y_train, y_test = y[:split_index], y[split_index:]

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)

Train size: (760, 7)
Test size: (191, 7)


## pipeline (TRAIN)

In [29]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', XGBClassifier(
        n_estimators=100,
        max_depth=5,
        learning_rate=0.1,
        use_label_encoder=False,
        eval_metric='logloss'
    ))
])

pipeline.fit(X_train, y_train)

d:\DATA SCIENCE\MACHINE LEARNING\Projects\MarketPulse-AI\venv\Lib\site-packages\xgboost\training.py:200: UserWarning: [10:38:14] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('scaler', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None


## Evaluate

In [30]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

y_pred = pipeline.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))

Accuracy: 0.5654450261780105
Precision: 0.65
Recall: 0.48598130841121495
F1 Score: 0.5561497326203209


## Save Model

In [31]:
import os
import pickle

# make sure folder exists
os.makedirs("artifacts", exist_ok=True)

# save the trained pipeline
with open("artifacts/xgb_pipeline.pkl", "wb") as f:
    pickle.dump(pipeline, f)

print("Model saved successfully")

Model saved successfully
